In [150]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error


data_path = 'data/housing.csv'
raw_df = pd.read_csv(data_path)
df = raw_df.copy()




In [151]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [152]:
print(df.info())
print(df['ocean_proximity'].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB
None
['NEAR BAY' '<1H OCEAN' 'INLAND' 'NEAR OCEAN' 'ISLAND']


In [153]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

In [154]:
df = pd.get_dummies(df, columns = ['ocean_proximity'], dtype=int)
df['RoomPerPerson'] = df['total_rooms'] / df['population']
df['IncPerAge'] = df['median_income'] / df['housing_median_age']
df['RoomPerHouse'] = df['total_rooms'] / df['households']
df['BedroomPercentage'] = df['total_bedrooms'] / df['total_rooms']


In [155]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN,RoomPerPerson,IncPerAge,RoomPerHouse,BedroomPercentage
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,0,0,0,1,0,2.732919,0.203054,6.984127,0.146591
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,0,0,0,1,0,2.956685,0.395305,6.238137,0.155797
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,0,0,0,1,0,2.957661,0.139565,8.288136,0.129516
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,0,0,0,1,0,2.283154,0.108521,5.817352,0.184458
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,0,0,0,1,0,2.879646,0.073965,6.281853,0.172096


In [156]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   longitude                   20640 non-null  float64
 1   latitude                    20640 non-null  float64
 2   housing_median_age          20640 non-null  float64
 3   total_rooms                 20640 non-null  float64
 4   total_bedrooms              20640 non-null  float64
 5   population                  20640 non-null  float64
 6   households                  20640 non-null  float64
 7   median_income               20640 non-null  float64
 8   median_house_value          20640 non-null  float64
 9   ocean_proximity_<1H OCEAN   20640 non-null  int64  
 10  ocean_proximity_INLAND      20640 non-null  int64  
 11  ocean_proximity_ISLAND      20640 non-null  int64  
 12  ocean_proximity_NEAR BAY    20640 non-null  int64  
 13  ocean_proximity_NEAR OCEAN  206

In [157]:
y = df['median_house_value'].to_numpy()[:2000]
X = df.drop(columns = ['median_house_value', 'total_rooms', 'total_bedrooms']).to_numpy()[:2000]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [158]:
class Node:
    def __init__(self, featureidx = None, threshold = None, left_node = None, right_node = None, value = None):
        self.featureidx = featureidx
        self.threshold = threshold
        self.left_node = left_node
        self.right_node = right_node


        self.value = value




In [159]:
class DecisionTree:
    def __init__(self, max_depth = 2, min_sample = 2):
        self.max_depth = max_depth
        self.min_sample = min_sample

    def build_tree(self, dataset, current_depth = 0):
        X, y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape
        
        if n_samples >= self.min_sample and current_depth < self.max_depth:
            best_split = self.best_split(dataset, n_features)

            if best_split['red'] > 1e-7:
                left_node = self.build_tree(best_split['left_node'], current_depth + 1)
                right_node = self.build_tree(best_split['right_node'], current_depth + 1)

                return Node(featureidx = best_split['featureidx'], threshold = best_split['threshold'], left_node = left_node, right_node = right_node)
            
        leaf_value = self.mean(y)
        return Node(value = leaf_value)


    def best_split(self, dataset, n_features):
        best_split = {'featureidx' : None, 'threshold' : None, 'red' : -1, 'left_node' : None, 'right_node' : None}

        for feature_idx in range(n_features):
            feature_values = dataset[:, feature_idx]
            values = np.sort(np.unique(feature_values))
            thresholds = (values[:-1] + values[1:]) / 2

            parent_y = dataset[:, -1]

            for threshold in thresholds:
                left_node, right_node = self.split(dataset, feature_idx, threshold)
                
                left_y, right_y = left_node[:, -1], right_node[:, -1]
                
                if len(left_node) < self.min_sample or len(right_node) < self.min_sample:
                    continue
                
                red = self.var_reduction(parent_y, left_y, right_y)

                if red > best_split['red']:
                    best_split['featureidx'] = feature_idx
                    best_split['threshold'] = threshold
                    best_split['left_node'] = left_node
                    best_split['right_node'] = right_node
                    best_split['red'] = red

        return best_split


    def split(self, dataset, feature_idx, threshold):
        left_mask = dataset[:, feature_idx] <= threshold
        right_mask = dataset[:, feature_idx] > threshold

        left_data = dataset[left_mask]
        right_data = dataset[right_mask]

        return left_data, right_data
    
    def mean(self, y):
        return np.mean(y)

    def mse(self, y):
        return np.mean((y - np.mean(y)) ** 2)


    def var_reduction(self, parent_y, left_y, right_y):
        weight_l = len(left_y) / len(parent_y)
        weight_r = len(right_y) / len(parent_y)

        return self.mse(parent_y) - (
            weight_l * self.mse(left_y) + 
            weight_r * self.mse(right_y)
        )
    
    def fit(self, X, y):
        dataset = np.concatenate([X, y.reshape(-1, 1)], axis = 1)
        self.root = self.build_tree(dataset)
    
    def predict(self, X):
        predictions = [self.predict_value(row, self.root) for row in X]
        return predictions
    
    def predict_value(self, row, node):
        if node.value != None:
            return node.value
        
        if row[node.featureidx] <= node.threshold:
            return self.predict_value(row, node.left_node)
        else:
            return self.predict_value(row, node.right_node)


            
        

        

        


In [160]:
dt = DecisionTree(5, 6)
dt.fit(X_train, y_train)


In [161]:
predictions = dt.predict(X_test)

print(predictions)


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


print(rmse(y_test, predictions))
print(mae(y_test, predictions))

[np.float64(98823.45679012345), np.float64(187772.78106508875), np.float64(149229.41176470587), np.float64(224030.0), np.float64(149229.41176470587), np.float64(114432.41758241758), np.float64(224030.0), np.float64(254241.86046511628), np.float64(122497.41450777202), np.float64(114432.41758241758), np.float64(177772.54901960783), np.float64(149229.41176470587), np.float64(98823.45679012345), np.float64(224030.0), np.float64(187772.78106508875), np.float64(284930.0), np.float64(122497.41450777202), np.float64(367924.44444444444), np.float64(500001.0), np.float64(98823.45679012345), np.float64(177772.54901960783), np.float64(263350.0), np.float64(197169.23076923078), np.float64(98823.45679012345), np.float64(114432.41758241758), np.float64(177772.54901960783), np.float64(68093.10344827586), np.float64(284930.0), np.float64(187772.78106508875), np.float64(122497.41450777202), np.float64(224030.0), np.float64(254241.86046511628), np.float64(187772.78106508875), np.float64(224030.0), np.flo

In [162]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=6,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 47580.47648774712
MAE: 35679.24770472554
